# IG-CAM-ViT -- Evaluation sur Vision Transformers (ISIC 2019)

**Methode** : Integrated Grad-CAM pour Vision Transformers (`ig_cam_vit.py`)

**Modeles** : ViT-Base/16 - DeiT-Base/16 - Swin-Base (BatchFormer)

**Selection** : high_confidence - medium_confidence - uncertain - incorrect

**Metriques** : Insertion AUC - Deletion AUC - Faithfulness - Completeness error


## 1. Imports

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
from PIL import Image
import torchvision.transforms as T
import timm
from pathlib import Path
from tqdm.notebook import tqdm
from scipy.stats import pearsonr
from scipy.ndimage import gaussian_filter
import warnings
warnings.filterwarnings('ignore')

from ig_cam_vit import IGCAMViT
from models.architectures import ViTBaseModel, DeiTBaseModel, SwinBaseModel

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {DEVICE}')
print(f'PyTorch: {torch.__version__}  |  timm: {timm.__version__}')

Device : cpu
PyTorch: 2.8.0+cpu  |  timm: 0.9.12


## 2. Configuration

In [ ]:
ISIC_IMG_DIR  = '../data/ISIC2019/ISIC_2019_Training_Input'
ISIC_CSV_PATH = '../data/ISIC2019/ISIC_2019_Training_GroundTruth.csv'
RESULTS_DIR   = Path('./igcam_results')
RESULTS_DIR.mkdir(exist_ok=True)
VIT_OUT = RESULTS_DIR / 'vit_evaluation'
VIT_OUT.mkdir(exist_ok=True)

NUM_CLASSES = 8
CLASS_NAMES = ['MEL', 'NV', 'BCC', 'AK', 'BKL', 'DF', 'VASC', 'SCC']

MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

N_PER_CATEGORY = 3
SAMPLE_SIZE    = 800
CONF_HIGH      = 0.80
CONF_MEDIUM    = 0.50
ENTROPY_FRAC   = 0.70
H_MAX = float(np.log(NUM_CLASSES))

N_STEPS       = 50
BATCH_SIZE_IG = 4
N_STEPS_AUC   = 100

MODEL_COLORS    = {'ViT-Base': '#e74c3c', 'DeiT-Base': '#3498db', 'Swin-Base': '#f39c12'}
STRATEGY_COLORS = {'S1-A': '#2ecc71', 'S1-B': '#27ae60', 'S2': '#3498db', 'S3': '#9b59b6'}

print('Configuration OK')

Configuration OK


## 3. Chargement des modeles

**Correction automatique de deux cas courants :**

- **(a)** Le checkpoint a ete sauvegarde depuis le backbone timm directement (cles sans `backbone.`). Le wrapper `BaseModel` utilise `self.backbone`, donc on ajoute ce prefixe.
- **(b)** Renommage timm Swin : `head.weight` devient `head.fc.weight` dans les versions recentes.


In [3]:
def load_vit_model(arch_class, ckpt_path, num_classes=NUM_CLASSES):
    model = arch_class(num_classes=num_classes, pretrained=False)
    ckpt  = torch.load(ckpt_path, map_location='cpu')

    # Extraire le state dict
    if isinstance(ckpt, dict):
        state = ckpt.get('model_state_dict',
                ckpt.get('state_dict', ckpt.get('model', ckpt)))
    else:
        state = ckpt

    # Supprimer prefixe DataParallel
    state = {k.replace('module.', ''): v for k, v in state.items()}

    # Cas (a) : ajouter backbone. si absent du checkpoint
    model_keys           = {k for k, _ in model.named_parameters()}
    ckpt_has_backbone    = any(k.startswith('backbone.') for k in state)
    model_needs_backbone = any(k.startswith('backbone.') for k in model_keys)
    if model_needs_backbone and not ckpt_has_backbone:
        print('  [INFO] Ajout prefixe backbone. (checkpoint sauvegarde sans wrapper)')
        state = {'backbone.' + k: v for k, v in state.items()}

    # Cas (b) : renommage Swin head (changement API timm)
    for old_k, new_k in [('backbone.head.weight', 'backbone.head.fc.weight'),
                          ('backbone.head.bias',   'backbone.head.fc.bias')]:
        if old_k in state and new_k not in state:
            state[new_k] = state.pop(old_k)
            print(f'  [INFO] Renommage {old_k} -> {new_k}')

    missing, _ = model.load_state_dict(state, strict=False)
    if missing:
        ok_missing = {'backbone.head.fc.weight', 'backbone.head.fc.bias',
                      'backbone.head.weight', 'backbone.head.bias'}
        critical = [k for k in missing if k not in ok_missing]
        if critical:
            print(f'  [WARN] Cles critiques manquantes ({len(critical)}): {critical[:4]}')
        else:
            print(f'  [OK]   Seules les cles head absentes : {missing}')
    else:
        print('  [OK]   Toutes les cles chargees')

    model.eval().to(DEVICE)
    return model


print('Chargement ViT-Base ...')
vit_model  = load_vit_model(ViTBaseModel,  '../results/vit_base_patch16_best.pth')

print('Chargement DeiT-Base ...')
deit_model = load_vit_model(DeiTBaseModel, '../results/deit_base_patch16_best.pth')

print('Chargement Swin-Base (BatchFormer) ...')
swin_model = load_vit_model(SwinBaseModel, '../results/swin_base_batchformer_best.pth')

# ViT/DeiT : tokens (B, 197, 768) avec CLS, grille 14x14
# Swin      : tokens (B, 49, 1024) sans CLS, grille 7x7
MODEL_REGISTRY = {
    'ViT-Base':  {'model': vit_model,  'target_block': vit_model.backbone.blocks[-1],
                  'has_cls_token': True,  'num_patches_side': 14, 'supports_attn': True},
    'DeiT-Base': {'model': deit_model, 'target_block': deit_model.backbone.blocks[-1],
                  'has_cls_token': True,  'num_patches_side': 14, 'supports_attn': True},
    'Swin-Base': {'model': swin_model, 'target_block': swin_model.backbone.layers[-1].blocks[-1],
                  'has_cls_token': False, 'num_patches_side': 7,  'supports_attn': False},
}
print(f'\n{len(MODEL_REGISTRY)} modeles charges.')

Chargement ViT-Base ...
  [INFO] Ajout prefixe backbone. (checkpoint sauvegarde sans wrapper)
  [INFO] Renommage backbone.head.weight -> backbone.head.fc.weight
  [INFO] Renommage backbone.head.bias -> backbone.head.fc.bias
  [OK]   Seules les cles head absentes : ['backbone.head.weight', 'backbone.head.bias']
Chargement DeiT-Base ...
  [INFO] Ajout prefixe backbone. (checkpoint sauvegarde sans wrapper)
  [INFO] Renommage backbone.head.weight -> backbone.head.fc.weight
  [INFO] Renommage backbone.head.bias -> backbone.head.fc.bias
  [OK]   Seules les cles head absentes : ['backbone.head.weight', 'backbone.head.bias']
Chargement Swin-Base (BatchFormer) ...
  [OK]   Seules les cles head absentes : ['backbone.head.fc.weight', 'backbone.head.fc.bias']

3 modeles charges.


## 4. Preprocessing & utilitaires

In [4]:
preprocess = T.Compose([T.Resize((224,224)), T.ToTensor(), T.Normalize(mean=MEAN, std=STD)])
viz_tf     = T.Compose([T.Resize((224,224)), T.ToTensor()])


def load_image(path):
    pil = Image.open(path).convert('RGB')
    return (preprocess(pil).unsqueeze(0).to(DEVICE),
            viz_tf(pil).permute(1,2,0).numpy(), pil)


def predict(model, tensor):
    model.eval()
    with torch.no_grad():
        probs = torch.softmax(model(tensor), dim=1)[0].cpu().numpy()
    cls = int(probs.argmax())
    return cls, float(probs[cls]), probs


def entropy_norm(probs):
    p = probs[probs > 0]
    return float(-np.sum(p * np.log(p)) / H_MAX)


def classify_image(conf, probs, true_cls=None, pred_cls=None):
    cats = []
    if conf >= CONF_HIGH:   cats.append('high_confidence')
    elif conf >= CONF_MEDIUM: cats.append('medium_confidence')
    if entropy_norm(probs) >= ENTROPY_FRAC: cats.append('uncertain')
    if true_cls is not None and pred_cls != true_cls and conf >= CONF_HIGH:
        cats.append('incorrect')
    return cats or ['medium_confidence']


def overlay_cam(img_np, saliency, alpha=0.5, cmap='jet'):
    heatmap = cm.get_cmap(cmap)(saliency)[:, :, :3]
    return np.clip(alpha * heatmap + (1 - alpha) * img_np, 0, 1)


print('Utilitaires OK')

Utilitaires OK


## 5. Selection multi-criteres des images de test

In [ ]:
# Charger le CSV ground truth
gt_df = pd.read_csv(ISIC_CSV_PATH)
img_col    = 'image' if 'image' in gt_df.columns else gt_df.columns[0]
label_cols = [c for c in gt_df.columns if c != img_col]
gt_df['true_cls'] = gt_df[label_cols].values.argmax(axis=1)
print(f'CSV : {len(gt_df)} lignes  |  classes : {gt_df["true_cls"].value_counts().sort_index().to_dict()}')


# ── Helpers ─────────────────────────────────────────────────────────────

def get_model_predictions(model_registry, tensor):
    """Retourne les predictions de tous les modeles du registre."""
    preds = {}
    for mname, cfg in model_registry.items():
        cls, conf, probs = predict(cfg['model'], tensor)
        preds[mname] = {'class': cls, 'confidence': conf, 'probs': probs}
    return preds


def select_evaluation_images(df, img_dir, model_registry, n_per_category=N_PER_CATEGORY):
    """
    Selectionne des images selon le consensus entre modeles.

    Categories (sans per_class) :
      high_consensus   : tous les modeles corrects, conf moyenne > 80%
      medium_consensus : majorite correcte, conf moyenne 50-80%
      disagreement     : les modeles ne sont pas d'accord entre eux
      difficult        : tous faux OU conf moyenne < 50%

    Retourne un dict {categorie: [image_info, ...]}
    et une liste plate selected_images pour la suite du notebook.
    """
    CATEGORIES = ['high_consensus', 'medium_consensus', 'disagreement', 'difficult']
    selected   = {c: [] for c in CATEGORIES}
    n_models   = len(model_registry)

    sample_df = df.sample(n=min(SAMPLE_SIZE, len(df)), random_state=42)

    for _, row in tqdm(sample_df.iterrows(),
                       total=len(sample_df), desc='Selection multi-modeles'):
        img_id   = row[img_col]
        true_cls = int(row['true_cls'])

        img_path = next(
            (Path(img_dir) / f'{img_id}{ext}'
             for ext in ['.jpg', '.jpeg', '.png', '.JPG']
             if (Path(img_dir) / f'{img_id}{ext}').exists()), None)
        if img_path is None:
            continue

        try:
            tensor, viz, _ = load_image(img_path)
            preds = get_model_predictions(model_registry, tensor)
        except Exception:
            continue

        # Metriques de consensus
        correct_count = sum(1 for p in preds.values() if p['class'] == true_cls)
        avg_conf      = float(np.mean([p['confidence'] for p in preds.values()]))
        pred_classes  = [p['class'] for p in preds.values()]
        all_agree     = len(set(pred_classes)) == 1
        # pred_cls = vote majoritaire parmi les modeles
        pred_cls_majority = max(set(pred_classes), key=pred_classes.count)

        entry = {
            'image_id':       img_id,
            'path':           str(img_path),
            'tensor':         tensor,
            'viz':            viz,
            'true_cls':       true_cls,
            'pred_cls':       pred_cls_majority,
            'confidence':     avg_conf,
            'correct_count':  correct_count,
            'agreement':      all_agree,
            'is_correct':     correct_count == n_models,
            'predictions':    preds,
        }

        # Categorisation selon le consensus multi-modeles
        if correct_count == n_models and avg_conf > 0.80:
            cat = 'high_consensus'
        elif correct_count >= n_models // 2 and 0.50 <= avg_conf <= 0.80:
            cat = 'medium_consensus'
        elif not all_agree:
            cat = 'disagreement'
        elif correct_count == 0 or avg_conf < 0.50:
            cat = 'difficult'
        else:
            continue  # aucune categorie applicable

        if len(selected[cat]) < n_per_category:
            entry['category'] = cat
            selected[cat].append(entry)

        # Arret des que tous les quotas sont remplis
        if all(len(selected[c]) >= n_per_category for c in CATEGORIES):
            break

    # Resume
    print('' + '='*60)
    print('IMAGES SELECTIONNEES :')
    print('='*60)
    for cat in CATEGORIES:
        imgs = selected[cat]
        print(f'  {cat:20s}: {len(imgs)}')
        for im in imgs:
            preds_str = '  '.join(
                f"{mn}={CLASS_NAMES[p['class']]}({p['confidence']:.0%})"
                for mn, p in im['predictions'].items()
            )
            cor = 'OK' if im['is_correct'] else 'ERR'
            print(f"    {im['image_id'][:14]:14s} GT={CLASS_NAMES[im['true_cls']]:4s} "
                  f"{cor}  {preds_str}")

    return selected


# ── Appel ───────────────────────────────────────────────────────────────
selected_by_cat = select_evaluation_images(
    gt_df, ISIC_IMG_DIR, MODEL_REGISTRY, n_per_category=N_PER_CATEGORY)

# Liste plate utilisee par le reste du notebook
selected_images = [
    img for imgs in selected_by_cat.values() for img in imgs
]
print(f'Total : {len(selected_images)} images')

# Mettre a jour les couleurs de categories pour la visualisation
CAT_COLORS = {
    'high_consensus':   '#27ae60',
    'medium_consensus': '#f39c12',
    'disagreement':     '#8e44ad',
    'difficult':        '#e74c3c',
}
cats_ordered = ['high_consensus', 'medium_consensus', 'disagreement', 'difficult']


CSV : 25331 lignes  |  classes : {0: 4522, 1: 12875, 2: 3323, 3: 867, 4: 2624, 5: 239, 6: 253, 7: 628}


Selection multi-modeles:   0%|          | 0/800 [00:00<?, ?it/s]

In [ ]:
groups = {c: [im for im in selected_images if im['category'] == c]
          for c in cats_ordered}
max_col = max((len(v) for v in groups.values()), default=1)

fig, axes = plt.subplots(len(cats_ordered), max_col + 1,
                          figsize=(3.5 * (max_col + 1), 3.5 * len(cats_ordered)))
if len(cats_ordered) == 1:
    axes = axes[np.newaxis, :]

CAT_LABELS = {
    'high_consensus':   'Consensus eleve (tous OK, conf > 80%)',
    'medium_consensus': 'Consensus moyen (majorite OK, 50-80%)',
    'disagreement':     'Desaccord entre modeles',
    'difficult':        'Cas difficiles (tous faux ou conf < 50%)',
}

for ri, cat in enumerate(cats_ordered):
    color = CAT_COLORS[cat]
    axes[ri, 0].axis('off')
    axes[ri, 0].text(0.5, 0.5, CAT_LABELS[cat],
                     ha='center', va='center', fontsize=8,
                     color=color, fontweight='bold', wrap=True)
    for ci, im in enumerate(groups[cat]):
        ax = axes[ri, ci + 1]
        ax.imshow(im['viz'])
        ax.axis('off')
        # Afficher pred de chaque modele
        preds_short = ' | '.join(
            f"{mn[:4]}={CLASS_NAMES[p['class']]}({p['confidence']:.0%})"
            for mn, p in im['predictions'].items()
        )
        cor = 'OK' if im['is_correct'] else 'ERR'
        ax.set_title(
            f"{im['image_id'][:12]}
GT={CLASS_NAMES[im['true_cls']]} {cor}
{preds_short}",
            fontsize=6, color=color
        )
    for ci in range(len(groups[cat]) + 1, max_col + 1):
        axes[ri, ci].axis('off')

plt.suptitle('Images selectionnees par consensus multi-modeles', fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig(VIT_OUT / 'selected_images.png', dpi=120, bbox_inches='tight')
plt.show()


## 6. Fonctions IG-CAM-ViT & metriques

In [ ]:
def get_igcam_vit(cfg, tensor, target_cls, variant='A', strategy='1',
                  n_steps=N_STEPS, batch_size=BATCH_SIZE_IG):
    if strategy in ('2','3') and not cfg['supports_attn']:
        raise ValueError(f'Strategy {strategy} requiert CLS token (non dispo pour Swin)')
    igcam = IGCAMViT(model=cfg['model'], target_block=cfg['target_block'],
                     n_steps=n_steps, variant=variant, strategy=strategy,
                     num_patches_side=cfg['num_patches_side'],
                     has_cls_token=cfg['has_cls_token'])
    try:
        cam = igcam.generate(tensor, target_class=target_cls, batch_size=batch_size)
    finally:
        igcam.remove_hooks()
    return cam


def get_igcam_completeness(cfg, tensor, target_cls, variant='A',
                            n_steps=N_STEPS, batch_size=BATCH_SIZE_IG):
    igcam = IGCAMViT(model=cfg['model'], target_block=cfg['target_block'],
                     n_steps=n_steps, variant=variant, strategy='1',
                     num_patches_side=cfg['num_patches_side'],
                     has_cls_token=cfg['has_cls_token'])
    try:
        r = igcam.verify_completeness(tensor, target_class=target_cls, batch_size=batch_size)
    finally:
        igcam.remove_hooks()
    return r


def insertion_deletion_auc(model, tensor, cam, target_cls,
                            n_steps=N_STEPS_AUC, blur_sigma=10.0):
    model.eval()
    img_np   = tensor.squeeze(0).cpu().numpy()
    C, H, W  = img_np.shape
    blurred  = np.stack([gaussian_filter(img_np[c], sigma=blur_sigma) for c in range(C)])
    idx_sort = np.argsort(cam.flatten())[::-1]
    step     = max(1, len(idx_sort)//n_steps)
    si, sd   = [], []
    with torch.no_grad():
        for i in range(0, len(idx_sort), step):
            idx  = idx_sort[:i+step]
            rows, cols = np.unravel_index(idx, (H,W))
            t = torch.tensor(blurred, dtype=torch.float32).unsqueeze(0).to(DEVICE)
            t[:,:,rows,cols] = tensor[:,:,rows,cols]
            si.append(torch.softmax(model(t), dim=1)[0, target_cls].item())
            t = tensor.clone()
            t[:,:,rows,cols] = torch.tensor(blurred[:,rows,cols], dtype=torch.float32).to(DEVICE)
            sd.append(torch.softmax(model(t), dim=1)[0, target_cls].item())
    si, sd = np.array(si), np.array(sd)
    auc_i, auc_d = float(np.trapz(si)/len(si)), float(np.trapz(sd)/len(sd))
    return {'auc_insertion': auc_i, 'auc_deletion': auc_d,
            'faithfulness': auc_i-auc_d, 'scores_ins': si, 'scores_del': sd}


print('Fonctions OK')

## 7. Evaluation -- boucle principale

- **S1-A** : tokens, GAP -- tous modeles
- **S1-B** : tokens, pixel-wise ReLU -- tous modeles
- **S2**   : attention CLS -- ViT/DeiT uniquement
- **S3**   : hybride tokens x attention -- ViT/DeiT uniquement


In [ ]:
METHODS = [
    {'key':'S1-A','variant':'A','strategy':'1','vit_only':False},
    {'key':'S1-B','variant':'B','strategy':'1','vit_only':False},
    {'key':'S2',  'variant':'A','strategy':'2','vit_only':True},
    {'key':'S3',  'variant':'B','strategy':'3','vit_only':True},
]

all_results  = []
maps_store   = {}
curves_store = {}

for img_info in tqdm(selected_images, desc='Images'):
    iid    = img_info['image_id']
    tensor = img_info['tensor']
    cat    = img_info['category']
    maps_store[iid]   = {}
    curves_store[iid] = {}

    for mname, cfg in MODEL_REGISTRY.items():
        pred_cls, conf, _ = predict(cfg['model'], tensor)
        maps_store[iid][mname]   = {}
        curves_store[iid][mname] = {}

        for m in METHODS:
            if m['vit_only'] and not cfg['supports_attn']: continue
            try:
                cam     = get_igcam_vit(cfg, tensor, pred_cls,
                                        variant=m['variant'], strategy=m['strategy'])
                metrics = insertion_deletion_auc(cfg['model'], tensor, cam, pred_cls)
                maps_store[iid][mname][m['key']]   = cam
                curves_store[iid][mname][m['key']] = {'ins': metrics['scores_ins'],
                                                       'del': metrics['scores_del']}
                compl_err = None
                if m['strategy'] == '1':
                    compl     = get_igcam_completeness(cfg, tensor, pred_cls, variant=m['variant'])
                    compl_err = compl['relative_error']
                all_results.append({
                    'image_id': iid, 'category': cat,
                    'model': mname, 'method': m['key'],
                    'pred_cls': pred_cls, 'confidence': conf,
                    'is_correct': pred_cls == img_info['true_cls'],
                    'auc_insertion': metrics['auc_insertion'],
                    'auc_deletion':  metrics['auc_deletion'],
                    'faithfulness':  metrics['faithfulness'],
                    'completeness_err': compl_err,
                })
            except Exception as e:
                print(f'  [ERR] {iid}/{mname}/{m["key"]}: {e}')

results_df = pd.DataFrame(all_results)
print(f'\nEvaluation terminee : {len(results_df)} entrees')
results_df.head()

## 8. Visualisation -- grille par image (modeles x strategies)

In [ ]:
COL_KEYS   = ['S1-A','S1-B','S2','S3']
COL_LABELS = ['S1-A\n(tokens GAP)','S1-B\n(tokens pixel)','S2\n(attention)','S3\n(hybride)']

for img_info in selected_images:
    iid = img_info['image_id']; viz = img_info['viz']; cat = img_info['category']
    n_rows = len(MODEL_REGISTRY); n_cols = 1 + len(COL_KEYS)
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(3.3*n_cols, 3.3*n_rows))
    if n_rows==1: axes=axes[np.newaxis,:]

    axes[0,0].set_title('Original', fontsize=9, fontweight='bold')
    for ci,lbl in enumerate(COL_LABELS):
        axes[0,ci+1].set_title(lbl, fontsize=9, fontweight='bold',
                                color=STRATEGY_COLORS.get(COL_KEYS[ci],'#333'))

    for ri, mname in enumerate(MODEL_REGISTRY):
        p_cls, p_conf, _ = predict(MODEL_REGISTRY[mname]['model'], img_info['tensor'])
        corr = 'OK' if p_cls==img_info['true_cls'] else 'ERR'
        axes[ri,0].set_ylabel(f"{mname}\n{CLASS_NAMES[p_cls]} {corr}\n({p_conf:.0%})",
                               fontsize=8, color=MODEL_COLORS.get(mname,'black'),
                               rotation=0, labelpad=90, va='center')
        axes[ri,0].imshow(viz); axes[ri,0].axis('off')
        for ci, key in enumerate(COL_KEYS):
            cam = maps_store[iid].get(mname,{}).get(key)
            if cam is not None:
                axes[ri,ci+1].imshow(overlay_cam(viz, cam))
            else:
                axes[ri,ci+1].imshow(viz, alpha=0.25)
                axes[ri,ci+1].text(0.5,0.5,'N/A',ha='center',va='center',
                                   transform=axes[ri,ci+1].transAxes,fontsize=13,color='gray')
            axes[ri,ci+1].axis('off')

    plt.suptitle(f"{iid}  |  GT: {CLASS_NAMES[img_info['true_cls']]}  |  [{cat}]",
                 fontsize=11, y=1.01)
    plt.tight_layout()
    plt.savefig(VIT_OUT/f'{iid}_all_models.png', dpi=110, bbox_inches='tight')
    plt.show(); plt.close()

## 9. Courbes Insertion / Deletion

In [ ]:
for img_info in selected_images[:min(3, len(selected_images))]:
    iid = img_info['image_id']; cat = img_info['category']
    fig, axes = plt.subplots(2, len(MODEL_REGISTRY), figsize=(5*len(MODEL_REGISTRY), 8))

    for ci, mname in enumerate(MODEL_REGISTRY):
        mmap = curves_store[iid].get(mname, {})
        for ri, (mode, title) in enumerate([('ins','Insertion (haut=bon)'),('del','Deletion (bas=bon)')]):
            ax = axes[ri,ci]
            for m in METHODS:
                if m['key'] not in mmap: continue
                curve = mmap[m['key']][mode]
                ax.plot(np.linspace(0,1,len(curve)), curve,
                        lw=1.8, color=STRATEGY_COLORS.get(m['key'],'#555'), label=m['key'])
            ax.set_title(f'{mname}\n{title}', fontsize=9); ax.legend(fontsize=8)
            ax.set_xlabel('Fraction pixels'); ax.grid(alpha=0.3)
            if ci==0: ax.set_ylabel('Score classe')

    plt.suptitle(f'{iid}  [{cat}]', fontsize=11)
    plt.tight_layout()
    plt.savefig(VIT_OUT/f'{iid}_ins_del_curves.png', dpi=110, bbox_inches='tight')
    plt.show(); plt.close()

## 10. Metriques : AUC & Faithfulness

In [ ]:
agg = (results_df.groupby(['model','method'])
       [['auc_insertion','auc_deletion','faithfulness']].mean().round(4))
print('=== AUC moyen par modele & methode ==='); print(agg.to_string())
agg.to_csv(VIT_OUT/'auc_by_model_method.csv')

pivot = results_df.groupby(['model','method'])['faithfulness'].mean().unstack('method')
fig, ax = plt.subplots(figsize=(8,4))
sns.heatmap(pivot, annot=True, fmt='.4f', cmap='RdYlGn', linewidths=0.5, ax=ax,
            cbar_kws={'label':'Faithfulness (Ins-Del)'})
ax.set_title('Faithfulness moyen par modele x strategie', fontsize=12)
plt.tight_layout()
plt.savefig(VIT_OUT/'faithfulness_heatmap.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
cats_order = ['high_consensus','medium_consensus','disagreement','difficult']
sub = results_df[results_df.method=='S1-A']
x, w = np.arange(len(cats_order)), 0.25
fig, ax = plt.subplots(figsize=(11,5))
for i, mname in enumerate(MODEL_REGISTRY):
    vals = [sub[(sub.model==mname)&(sub.category==c)]['faithfulness'].mean() for c in cats_order]
    bars = ax.bar(x+i*w, vals, w, label=mname,
                  color=MODEL_COLORS.get(mname,'#888'), alpha=0.85, edgecolor='white')
    for bar, v in zip(bars, vals):
        if not np.isnan(v):
            ax.text(bar.get_x()+w/2, bar.get_height()+0.002, f'{v:.3f}',
                    ha='center', va='bottom', fontsize=7)
ax.set_xticks(x+w); ax.set_xticklabels([c.replace('_','\n') for c in cats_order], fontsize=9)
ax.set_ylabel('Faithfulness'); ax.set_title('Faithfulness S1-A par categorie x modele', fontsize=12)
ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(VIT_OUT/'faithfulness_by_category.png', dpi=120, bbox_inches='tight')
plt.show()

## 11. Verification completude (axiome IG)

In [ ]:
compl_rows = []
for img_info in tqdm(selected_images, desc='Completeness'):
    tensor = img_info['tensor']; iid = img_info['image_id']
    for mname, cfg in MODEL_REGISTRY.items():
        pred_cls, _, _ = predict(cfg['model'], tensor)
        for variant in ['A','B']:
            try:
                r = get_igcam_completeness(cfg, tensor, pred_cls, variant=variant)
                compl_rows.append({'image_id':iid, 'category':img_info['category'],
                                   'model':mname, 'variant':f'S1-{variant}',
                                   'attr_sum':r['attribution_sum'],
                                   'output_diff':r['output_diff'],
                                   'rel_error':r['relative_error']})
            except Exception as e:
                print(f'  [ERR] {iid}/{mname}/S1-{variant}: {e}')

compl_df = pd.DataFrame(compl_rows)
compl_df['err_pct'] = compl_df['rel_error'] * 100
compl_df['verdict'] = compl_df['rel_error'].apply(
    lambda e: 'PASS' if e<0.05 else ('WARN' if e<0.20 else 'FAIL'))
print(compl_df.groupby(['model','variant'])['err_pct'].mean().round(2).to_string())
print(compl_df.groupby(['model','variant','verdict']).size().to_string())
compl_df.to_csv(VIT_OUT/'completeness_results.csv', index=False)

In [ ]:
fig, axes = plt.subplots(1, len(MODEL_REGISTRY),
                          figsize=(5*len(MODEL_REGISTRY),4), sharey=True)
for ax, mname in zip(axes, MODEL_REGISTRY):
    sub = compl_df[compl_df.model==mname]
    for vi, variant in enumerate(['S1-A','S1-B']):
        rows = sub[sub.variant==variant].reset_index()
        x_pos = [i+vi*0.35 for i in range(len(rows))]
        colors = ['#2ecc71' if e<0.05 else '#f39c12' if e<0.20 else '#e74c3c'
                  for e in rows['rel_error']]
        ax.bar(x_pos, rows['err_pct'].values, 0.32,
               color=colors, alpha=0.85, label=variant, edgecolor='white')
    ax.axhline(5,  color='orange', ls='--', lw=1.2, label='5%')
    ax.axhline(20, color='red',    ls='--', lw=1.2, label='20%')
    ax.set_title(mname, fontsize=10); ax.set_xlabel('Image'); ax.set_ylabel('Erreur (%)')
    ax.legend(fontsize=7); ax.grid(axis='y', alpha=0.3)
plt.suptitle('Verification completude IG-CAM-ViT (Strategy 1)', fontsize=12)
plt.tight_layout()
plt.savefig(VIT_OUT/'completeness_verification.png', dpi=120, bbox_inches='tight')
plt.show()

## 12. Correlation Pearson inter-strategies

In [ ]:
pearson_rows = []
for img_info in selected_images:
    iid = img_info['image_id']
    for mname in MODEL_REGISTRY:
        mmap = maps_store[iid].get(mname,{}); keys = list(mmap.keys())
        for i in range(len(keys)):
            for j in range(i+1,len(keys)):
                k1,k2 = keys[i],keys[j]
                c1,c2 = mmap[k1].flatten(), mmap[k2].flatten()
                if np.std(c1)>1e-8 and np.std(c2)>1e-8:
                    r,_ = pearsonr(c1,c2)
                    pearson_rows.append({'image_id':iid,'model':mname,
                                         'pair':f'{k1} vs {k2}','pearson_r':float(r)})

pearson_df = pd.DataFrame(pearson_rows)
if not pearson_df.empty:
    pairs = pearson_df['pair'].unique(); x,w = np.arange(len(pairs)), 0.25
    fig, ax = plt.subplots(figsize=(10,4))
    for i, mname in enumerate(MODEL_REGISTRY):
        sub  = pearson_df[pearson_df.model==mname]
        vals = [sub[sub.pair==p]['pearson_r'].mean() for p in pairs]
        ax.bar(x+i*w, vals, w, label=mname,
               color=MODEL_COLORS.get(mname,'#888'), alpha=0.85, edgecolor='white')
    ax.set_xticks(x+w); ax.set_xticklabels(pairs, rotation=20, fontsize=8)
    ax.axhline(0.98, color='red', ls='--', lw=1.2, label='r=0.98')
    ax.set_ylabel('Pearson r'); ax.set_title('Correlation inter-strategies IG-CAM-ViT')
    ax.legend(fontsize=8); ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig(VIT_OUT/'pearson_strategy_correlation.png', dpi=120, bbox_inches='tight')
    plt.show()

## 13. Recapitulatif global

In [ ]:
print('='*65)
print('RECAPITULATIF -- IG-CAM-ViT sur ISIC 2019')
print('='*65)
print(f'  Images : {len(selected_images)}  |  Modeles : {list(MODEL_REGISTRY.keys())}')
print(f'  N steps IG : {N_STEPS}  |  N steps AUC : {N_STEPS_AUC}')

print('\n-- Faithfulness moyen (S1-A) --')
for mname in MODEL_REGISTRY:
    sub = results_df[(results_df.model==mname)&(results_df.method=='S1-A')]
    if not sub.empty:
        print(f'  {mname:12s}: ins={sub["auc_insertion"].mean():.4f}  '
              f'del={sub["auc_deletion"].mean():.4f}  '
              f'faith={sub["faithfulness"].mean():.4f}')

if 'compl_df' in dir() and not compl_df.empty:
    print('\n-- Erreur completude moyenne (%) --')
    for mname in MODEL_REGISTRY:
        sub = compl_df[(compl_df.model==mname)&(compl_df.variant=='S1-A')]
        if not sub.empty: print(f'  {mname:12s}: {sub["err_pct"].mean():.2f}%')

results_df.to_csv(VIT_OUT/'all_results.csv', index=False)
print('\nCSV sauvegarde : all_results.csv')
print('\n-- Fichiers generes --')
for f in sorted(VIT_OUT.iterdir()): print(f'  {f.name}')